# PharmaPulse ML and Analytics Layer

This notebook reviews the explainable ML and analytics outputs created by `scripts/ml_pipeline.py`. The outputs are designed as segmentation, risk signal, forecast baseline, and prioritization score artifacts for commercial analytics workflows.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

cluster_profiles = pd.read_csv(PROCESSED_DIR / "cluster_profiles.csv")
hcp_segments = pd.read_csv(PROCESSED_DIR / "hcp_segments.csv")
risk_scores = pd.read_csv(PROCESSED_DIR / "hcp_churn_scores.csv")
sales_forecast = pd.read_csv(PROCESSED_DIR / "sales_forecast.csv")
next_best_action = pd.read_csv(PROCESSED_DIR / "next_best_action.csv")
master_scores = pd.read_csv(PROCESSED_DIR / "hcp_master_scores.csv")

print("Loaded processed outputs")
print(f"HCP master rows: {len(master_scores):,}")

## 1. HCP Segmentation

Business purpose: group HCPs into practical commercial segments using revenue, call activity, engagement, and call recency. These segments are directional planning groups, not fixed customer labels.

In [ ]:
cluster_profiles

In [ ]:
plot_data = cluster_profiles.set_index("segment_label")[[
    "mean_total_net_sales",
    "mean_total_calls",
    "mean_avg_engagement_score",
    "mean_days_since_last_call",
]]

scaled_plot_data = plot_data / plot_data.max()
ax = scaled_plot_data.plot(kind="bar", figsize=(10, 5), colormap="tab10")
ax.set_title("Cluster Profile Comparison (Scaled Metrics)")
ax.set_xlabel("Segment")
ax.set_ylabel("Scaled value")
ax.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

Interpretation: the segments separate HCPs by commercial value, engagement, and recency. The scaled comparison helps show the shape of each segment without letting revenue dominate the chart.

## 2. HCP Disengagement Risk Signal

Business purpose: create a transparent risk signal for follow-up planning. Because there are no observed churn labels, the risk level is rule-based and should be used as a prioritization proxy.

In [ ]:
risk_counts = risk_scores["risk_level"].value_counts().reindex([
    "High Risk",
    "Medium Risk",
    "Low Risk",
])
risk_counts

In [ ]:
ax = risk_counts.plot(kind="bar", figsize=(7, 4), colormap="tab10")
ax.set_title("HCP Risk Level Counts")
ax.set_xlabel("Risk level")
ax.set_ylabel("HCP count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Interpretation: the risk output highlights HCPs where call recency and engagement may require review. It is a directional risk signal for planning, not a churn prediction model.

## 3. Sales Forecast Baseline

Business purpose: provide a simple three-month revenue forecast baseline with a directional range. The band is a planning range, not a statistical confidence interval.

In [ ]:
sales_forecast.tail(8)

In [ ]:
forecast_plot = sales_forecast.copy()
forecast_plot["month"] = pd.to_datetime(forecast_plot["month"])

historical = forecast_plot[forecast_plot["record_type"] == "Historical"]
forecast = forecast_plot[forecast_plot["record_type"] == "Forecast"]

fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.get_cmap("viridis")
ax.plot(historical["month"], historical["actual_revenue"], marker="o", label="Actual revenue", color=cmap(0.25))
ax.plot(forecast["month"], forecast["forecast_revenue"], marker="o", label="Forecast baseline", color=cmap(0.75))
ax.fill_between(
    forecast["month"],
    forecast["forecast_lower"],
    forecast["forecast_upper"],
    alpha=0.2,
    color=cmap(0.75),
    label="Directional range",
)
ax.set_title("Monthly Sales Forecast Baseline")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Interpretation: the forecast provides a simple baseline for planning conversations. More advanced forecasting would require additional business drivers and validation against future actuals.

## 4. Next-Best-Action Prioritization

Business purpose: rank HCPs using revenue potential, call recency, engagement, and segment assignment so field teams can review follow-up priorities.

In [ ]:
top_actions = next_best_action.head(10)[[
    "recommendation_rank",
    "hcp_name",
    "territory_name",
    "segment_label",
    "risk_level",
    "prioritization_score",
    "recommended_action",
]]
top_actions

In [ ]:
top_plot = next_best_action.head(10).sort_values("prioritization_score")
ax = top_plot.plot(
    kind="barh",
    x="hcp_name",
    y="prioritization_score",
    figsize=(9, 5),
    legend=False,
    colormap="viridis",
)
ax.set_title("Top HCP Prioritization Scores")
ax.set_xlabel("Prioritization score")
ax.set_ylabel("HCP")
plt.tight_layout()
plt.show()

Interpretation: the highest-ranked HCPs combine stronger commercial value, recency need, engagement, and segment priority. Recommendations should be reviewed with territory context before action.

## 5. Validation Summary

Business purpose: confirm that the generated HCP intelligence outputs are complete enough for dashboard or Streamlit use.

In [ ]:
validation_summary = {
    "segments_have_no_missing_hcp_ids": hcp_segments["hcp_id"].notna().all(),
    "segment_labels_populated": hcp_segments["segment_label"].notna().all(),
    "cluster_profiles_has_3_rows": len(cluster_profiles) == 3,
    "risk_levels_populated": risk_scores["risk_level"].notna().all(),
    "forecast_has_historical_rows": (sales_forecast["record_type"] == "Historical").any(),
    "forecast_has_forecast_rows": (sales_forecast["record_type"] == "Forecast").any(),
    "recommendations_populated": next_best_action["recommended_action"].notna().all(),
    "master_has_500_rows": len(master_scores) == 500,
    "master_has_one_row_per_hcp": master_scores["hcp_id"].nunique() == 500,
}
pd.Series(validation_summary, name="passed")

Interpretation: these checks help confirm that the processed outputs are ready for lightweight reporting and prioritization workflows.